# Week 4: Supervised Learning & Regression
## Notebook 08: Quantile Regression & Pinball Loss from Scratch

### 1. Conceptual Intuition & Theoretical Background
Ordinary Least Squares (OLS) minimizes the mean squared error (MSE), modeling the **conditional mean** $\mathbb{E}[Y | X]$. However, OLS fails when:
1. Errors are **heteroscedastic** (variance grows with $X$).
2. The data contains severe outliers.
3. We need predictive intervals (e.g. 10th and 90th percentiles for financial Value-at-Risk).

**Quantile Regression** models the $\tau$-th conditional quantile $Q_Y(\tau | X) = X\beta$ by minimizing the **Pinball Loss (Asymmetric Absolute Loss)**:

$$\rho_\tau(u) = u \cdot (\tau - \mathbb{I}(u < 0)) = \begin{cases} \tau \cdot u & \text{if } u \ge 0 \\ (\tau - 1) \cdot u & \text{if } u < 0 \end{cases}$$

Where residual $u = y - X\beta$.
Subgradient with respect to $\beta$:
$$\frac{\partial \rho_\tau}{\partial \beta} = -X^T (\tau - \mathbb{I}(y - X\beta < 0))$$

---

### 2. Scratch Python Implementation: Quantile Regressor

In [ ]:
import numpy as np

class QuantileRegressorScratch:
    def __init__(self, tau=0.5, lr=0.01, n_iter=2000):
        self.tau = tau
        self.lr = lr
        self.n_iter = n_iter
        self.weights = None
        self.bias = 0.0
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = np.median(y)
        
        for epoch in range(self.n_iter):
            preds = np.dot(X, self.weights) + self.bias
            residuals = y - preds
            
            # Subgradient vector: tau if residual >= 0 else (tau - 1)
            subgrad = np.where(residuals >= 0, self.tau, self.tau - 1.0)
            
            # Gradient updates
            dw = -np.dot(X.T, subgrad) / n_samples
            db = -np.mean(subgrad)
            
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
        return self
        
    def predict(self, X):
        return np.dot(X, self.weights) + self.bias

# Synthetic Heteroscedastic Dataset
np.random.seed(42)
X_data = np.linspace(1, 10, 200).reshape(-1, 1)
# Noise standard deviation increases linearly with X
noise = np.random.normal(0, 0.4 * X_data.squeeze(), size=200)
y_data = 2.5 * X_data.squeeze() + 5.0 + noise

# Fit 10th (lower bound), 50th (median), and 90th (upper bound) percentiles
model_p10 = QuantileRegressorScratch(tau=0.10, lr=0.02, n_iter=3000).fit(X_data, y_data)
model_p50 = QuantileRegressorScratch(tau=0.50, lr=0.02, n_iter=3000).fit(X_data, y_data)
model_p90 = QuantileRegressorScratch(tau=0.90, lr=0.02, n_iter=3000).fit(X_data, y_data)

pred_10 = model_p10.predict(X_data)
pred_50 = model_p50.predict(X_data)
pred_90 = model_p90.predict(X_data)

# Verify empirical coverage of 80% prediction interval [P10, P90]
in_interval = np.mean((y_data >= pred_10) & (y_data <= pred_90)) * 100
print(f"Fitted Median Slope (tau=0.50):    {model_p50.weights[0]:.4f} (True: 2.50)")
print(f"Empirical Coverage of [P10, P90]: {in_interval:.1f}% (Expected: ~80.0%)")

### 3. Step-by-Step Code Walkthrough
1. **Pinball Loss Subgradient**: When residual is positive, penalty gradient is $\tau$; when negative, penalty gradient is $\tau - 1$.
2. **Median Regression ($\tau = 0.5$)**: Equivalent to Least Absolute Deviations ($L_1$ regression), producing predictions robust against extreme outliers.
3. **Prediction Interval**: The band between $\tau = 0.10$ and $\tau = 0.90$ naturally widens as variance increases.

---

### 4. Industry Applications & Trade-offs
- **Demand Forecasting & Inventory:** Optimizing stock levels according to safety stock service level targets (e.g. 95th percentile peak demand).
- **Trade-off:** Quantile crossing can occur where individual models trained separately produce $\hat{y}_{0.90} < \hat{y}_{0.10}$ in unconstrained settings.

---

### 5. Top Interview Questions & Answers
**Q1: What is the Pinball Loss and why is it asymmetric?**
> **Answer:** Pinball loss weights positive and negative errors differently depending on the quantile $\tau$. For $\tau = 0.9$, an underprediction ($y > \hat{y}$) is penalized 9 times more severely than an overprediction.

**Q2: What is the "Quantile Crossing" problem and how is it mitigated?**
> **Answer:** Quantile crossing occurs when independent models predict a higher quantile below a lower quantile. It is mitigated by multi-output joint networks with monotonic constraints or non-crossing penalties.